<a href="https://colab.research.google.com/github/arulbenjaminchandru/ai-architect-program/blob/main/Day_16.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Day 8 — Build a Goal-Driven Claude Agent (Plans Steps & Uses Tools)
### AI Architect Mastery Program · Duration ~2 hours · Level: Intelligent Beginner

Today you build an **autonomous agent**: you give it *one goal*, and it figures out the steps, calls tools to do the work, checks the results, and keeps going until the job is done — without you steering each step.

You will build it **two ways**, so you understand it inside-out:
1. **By hand, with the Claude API** — you write the agent loop in plain Python. This shows you exactly how an agent thinks and acts.
2. **With Claude Code** — Anthropic's ready-made command-line agent. Same loop, productionised, acting on real files.


**What you'll be able to do by the end**
- **Explain** what makes something an *agent* (vs. a single chatbot reply).
- **Demonstrate** the agent loop: *perceive → plan → act → observe → repeat*.
- **Implement** tools and the tool-use loop with the Claude API.
- **Build** an agent that completes a real multi-step task on its own.
- **Architect** it safely with limits, permissions, and least privilege.

**How each section is structured**

| Block | What it gives you |
|---|---|
| The simple questions | What / why / how / when (and when not) |
| Real-world example | Where this is actually used |
| Remember this | One interview-ready line |
| Don't mix these up | Quick wrong → right fixes |
| Lab | Short real code you run against the live Claude API |
| Interview + Quick check | Say it out loud, then test yourself |

---

# Foundations — From a Single Reply to an Agent

Before we build, let's place today's topic on the map and define the words we'll reuse all session.

**A plain LLM call** is one question in, one answer out. You ask Claude something, it replies, and it's done. It cannot look anything up, run anything, or take a second step. It only produces text.

**An agent** wraps that same model in a *loop* and gives it *tools*. Now the model can decide "I need to look something up," call a tool to do it, read the result, and decide what to do next — over and over — until the goal is met.

The five words we'll use constantly:

- **Goal** — what you want done, in plain language ("restock the low items").
- **Tool** — a function the agent is allowed to call (read a file, query stock, send an email).
- **Plan** — the agent's intended sequence of steps to reach the goal.
- **Act** — the agent actually calls a tool.
- **Observe** — the agent reads what the tool returned, then decides the next move.

> **DBA analogy :** a plain LLM call is like running a single `SELECT` and getting one result set back. An agent is more like a stored procedure with a **loop** inside — it fetches, checks a condition, does the next thing, and only `RETURN`s when the work is actually finished.


---

# Section 1 — What Is a Goal-Driven Agent?

**What is it?**
A goal-driven agent is an LLM running inside a loop, with permission to use tools, working toward a goal you stated once. You don't tell it each step — it decides the steps itself.

**Why does it matter?**
Most real work is *multi-step*: read data, compute something, write a result, check it. A single chatbot reply can't do that. An agent can chain steps and recover from mistakes, which is what turns a "smart text box" into something that actually completes tasks.

**How does it work? (the loop)**
```
        ┌─────────────────────────────────────────────┐
        │                                             │
   GOAL ─▶ PERCEIVE ─▶ PLAN/THINK ─▶ ACT (call tool) ─▶ OBSERVE
        │   (read goal,                  result            │
        │    last result)                                  │
        └──────────────── repeat until goal met ──────────┘
                                  │
                                DONE
```
The agent keeps looping. Each turn it looks at what it knows, decides the next action, does it, and reads the outcome — stopping only when the goal is reached.

**Where is it used?** Coding assistants, customer-support agents that look up orders, data-analysis agents, "research" agents that gather and summarise.

**When should I use one?** When the task needs several steps, external data, or self-correction.

**When should I avoid it?** When a single prompt already answers the question. An agent adds cost, latency, and risk — don't loop if you don't need to.

> **Accuracy note:** an agent is *not* a different, smarter model. It's the **same** Claude model, given a loop and tools. The intelligence is in the model; the *autonomy* comes from the loop you wrap around it.


**Real-world example.** GitHub Copilot's coding agent and Anthropic's own Claude Code take one goal ("fix the failing test") and autonomously read files, edit code, run the tests, and retry — the exact loop above.

> **Remember this:** *An agent is just an LLM in a loop with tools. The model decides; the loop lets it keep going.*

**Don't mix these up**
- ❌ "An agent is a special, more powerful model." → ✅ It's the same model wrapped in a loop with tools.
- ❌ "An agent plans everything up front, then runs." → ✅ It re-plans every turn based on what it just observed.
- ❌ "More tools always means a better agent." → ✅ Fewer, well-chosen tools are safer and more reliable.


**Interview question (beginner).**
*Q: What's the difference between a chatbot and an agent?*
A: A chatbot answers in one shot. An agent runs the model in a loop with tools, so it can take multiple actions, read results, and self-correct until a goal is met.

**Interview question (architecture).**
*Q: When would you NOT build an agent?*
A: When a single prompt or a fixed pipeline already solves it. Agents add cost, latency, and unpredictability; use the simplest thing that works.

## ✅ Quick Check 1
> **Q:** What two things turn a plain LLM call into an agent?
> **A:** A **loop** (so it can keep going) and **tools** (so it can act on the world).


## 🧪 Lab A — Set Up the Live Claude API

**Objective:** install the SDK and make your first real Claude call. (Run these in Colab or Jupyter with your own key.)

In [1]:
# cell 1 — install the Anthropic SDK
!pip install anthropic

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 929.8/929.8 kB 15.7 MB/s eta 0:00:00


In [2]:
# cell 2 — key + client (Colab Secret named MY_API_KEY)
from google.colab import userdata
import os, json
os.environ["ANTHROPIC_API_KEY"] = userdata.get("MY_API_KEY")

import anthropic
client = anthropic.Anthropic()
MODEL = "claude-haiku-4-5-20251001"   # fast + cheap, the right default for labs
print("Ready ✅")

Ready ✅


In [3]:
# cell 3 — your first call: one question in, one answer out (NOT yet an agent)
reply = client.messages.create(
    model=MODEL,
    max_tokens=200,
    messages=[{"role": "user", "content": "In one sentence, what is an AI agent?"}],
)
print(reply.content[0].text)
print("stop_reason:", reply.stop_reason)   # 'end_turn' = Claude finished talking

An AI agent is a system that perceives its environment, makes decisions based on that information, and takes actions to achieve specific goals.
stop_reason: end_turn


**Expected result:** a one-sentence definition, and `stop_reason: end_turn`.
**Try this:** add `system="You are a precise Oracle DBA mentor."` to the call and ask the same question — notice the flavour change.

> Right now this is *not* an agent: there's no loop and no tools. Next we add both.

---

# Section 2 — Tools: Giving the Agent Hands

**What is it?**
A **tool** is a function you let the agent call. You describe it to Claude as a small dictionary with a **name**, a **description** (when to use it), and an **input_schema** (what arguments it takes). Claude never runs your code directly — it *asks* to call the tool, and **your** code runs it.

**Why does it matter?**
On its own, a model can only produce text. Tools are what let it *do* things — read a file, look up stock, send an email. No tools, no actions.

**How does it work?**
1. You define the tools and pass them to `client.messages.create(..., tools=tools)`.
2. When Claude wants one, it replies with `stop_reason == "tool_use"` and the arguments it chose.
3. Your code runs the real function and sends the result back.
4. Claude reads the result and continues.

**When to use / avoid:** give the agent exactly the tools the task needs — no more. Extra tools confuse the model and widen what can go wrong.

> **DBA analogy:** a tool definition is like a **stored procedure signature** — a clear name, a description of what it does, and typed parameters (`input_schema`). Claude "calls" it the way an app calls a stored proc; the database (your code) is what actually executes it.


**Real-world example.** A banking support agent gets tools like `lookup_account`, `list_transactions`, and `open_dispute`. Claude decides *which* to call and *with what arguments*; the bank's backend executes them.

> **Remember this:** *A tool is a function with a name, a description, and a typed input. Claude chooses it; your code runs it.*

**Don't mix these up**
- ❌ "Claude runs my tool function itself." → ✅ Claude only *requests* the call; your code executes it and returns the result.
- ❌ "The description is just a comment." → ✅ The description is how Claude decides *when* to use the tool — write it carefully.


## 🧪 Lab B — Define a Tool and Inspect It

**Objective:** see exactly what a tool definition looks like. This cell is pure Python — it runs anywhere, no API needed.

In [4]:
# A tool is just a dictionary. Name + description + typed inputs.
tools = [{
    "name": "check_stock",
    "description": "Get how many units of a given item are currently in stock.",
    "input_schema": {
        "type": "object",
        "properties": {"item": {"type": "string", "description": "The product name, e.g. 'milk'"}},
        "required": ["item"],
    },
}]

import json
print(json.dumps(tools[0], indent=2))
print("\nThink of this as a stored-procedure signature Claude is allowed to call.")

{
  "name": "check_stock",
  "description": "Get how many units of a given item are currently in stock.",
  "input_schema": {
    "type": "object",
    "properties": {
      "item": {
        "type": "string",
        "description": "The product name, e.g. 'milk'"
      }
    },
    "required": [
      "item"
    ]
  }
}

Think of this as a stored-procedure signature Claude is allowed to call.


**Expected result:** the tool printed as clean JSON.
**Try this:** add a second tool `place_reorder(item, quantity)` to the list with the right schema.

---

# Section 3 — The Agent Loop with the Claude API

**What is it?**
The agent loop is the short piece of code that runs Claude over and over: ask the model, if it wants a tool then run the tool and send the result back, repeat — until it stops asking for tools.

**Why does it matter?**
This loop *is* the agent. Once you can write these ~10 lines, you understand every agent framework, because they all do this underneath.

**How does it work? The one rule:**
> **While `stop_reason == "tool_use"`: run the tool, send the result back, loop again.**

When `stop_reason` becomes `"end_turn"`, Claude has finished and you print its final answer. You also add a **turn limit** so a confused agent can't loop forever.

> **DBA analogy :** the loop is a **cursor `FETCH` loop**. You keep fetching the next action and processing it until there's nothing left to do (`end_turn`), and the turn limit is your safety valve against a runaway process.


**Real-world example.** Every "use the tools to answer" feature — order lookups, itinerary builders, data assistants — runs this exact loop on the server.

> **Remember this:** *The whole agent is one rule: while the model asks for a tool, run it and feed the result back.*

**Don't mix these up**
- ❌ "I send the tool result as a new user question." → ✅ You append it as a `tool_result` block tied to the tool's `id`, so Claude knows which call it answers.
- ❌ "The loop runs until I'm satisfied." → ✅ It runs until `stop_reason != "tool_use"` (or you hit the turn limit).


## 🧪 Lab C — The Full Agent Loop (Dedicated Claude API Lab)

**Objective:** build a real agent that answers a question by calling a tool, using the live API end-to-end. This is the heart of the session — run it and watch the loop work.

First, the tools and the real functions behind them:

In [5]:
# 1) The data our tools work on (pretend this is a tiny inventory database)
STOCK = {"milk": 8, "bread": 3, "eggs": 40, "butter": 2, "rice": 25}

# 2) The tool definitions Claude can see
tools = [
    {
        "name": "check_stock",
        "description": "Get how many units of an item are currently in stock.",
        "input_schema": {
            "type": "object",
            "properties": {"item": {"type": "string"}},
            "required": ["item"],
        },
    },
    {
        "name": "list_items",
        "description": "List all item names available in the inventory.",
        "input_schema": {"type": "object", "properties": {}},
    },
]

# 3) The real code that runs when Claude requests a tool (your "stored procedures")
def run_tool(name, args):
    if name == "check_stock":
        item = args["item"].lower()
        return STOCK.get(item, f"No item named {item}")
    if name == "list_items":
        return list(STOCK.keys())
    return f"Unknown tool: {name}"

print("Tools and runner ready ✅")

Tools and runner ready ✅


In [6]:
# The agent loop — this is the whole agent in ~12 lines
messages = [{"role": "user",
             "content": "Which costs me more attention: how much milk and butter do I have? Use the tools."}]

for turn in range(5):                      # turn limit = safety valve (no runaway loops)
    reply = client.messages.create(model=MODEL, max_tokens=500, tools=tools, messages=messages)
    messages.append({"role": "assistant", "content": reply.content})

    if reply.stop_reason != "tool_use":    # Claude is done thinking → print its answer
        print("FINAL:", reply.content[0].text)
        break

    results = []
    for block in reply.content:            # Claude may ask for several tools at once
        if block.type == "tool_use":
            print(f"  ACT  → {block.name}({block.input})")
            output = run_tool(block.name, block.input)
            print(f"  OBS  ← {output}")
            results.append({"type": "tool_result", "tool_use_id": block.id, "content": str(output)})

    messages.append({"role": "user", "content": results})   # OBSERVE: send results back

  ACT  → check_stock({'item': 'milk'})
  OBS  ← 8
  ACT  → check_stock({'item': 'butter'})
  OBS  ← 2
FINAL: Based on the stock check results:

- **Milk**: 8 units
- **Butter**: 2 units

**Butter costs you more attention.** With only 2 units in stock compared to 8 units of milk, butter is running significantly lower and would likely need to be restocked sooner. You should keep a closer eye on your butter supply.


**Expected result:** you'll see the agent ACT (call `check_stock` for milk and butter), OBSERVE the numbers, then print a FINAL sentence using them. That's *perceive → act → observe → repeat → done*.

**Try this:**
1. Ask: *"Which items are below 10? Use the tools."* — watch it call `list_items` first, then `check_stock` several times. That's multi-step planning.
2. Add a `place_reorder(item, quantity)` tool plus its runner, and ask it to restock everything below 10.

> **Cost & safety note:** you pay per token, so keep `max_tokens` small (Haiku is cheap). For any tool that *changes* something (reorders, payments), add a human "ok? y/n" check before `run_tool` runs it.

## ✅ Quick Check 2
> **Q:** In the loop, what tells your code to run a tool versus to stop?
> **A:** `reply.stop_reason`. If it's `"tool_use"`, run the requested tool(s) and loop; anything else (e.g. `"end_turn"`) means stop and print the answer.

---

# Section 4 — Planning: How the Agent Breaks a Goal into Steps

**What is it?**
Planning is the agent deciding the *order* of steps to reach a goal. With Claude you get this two ways: **implicit** planning (the model just picks the next tool each turn) and **explicit** planning (you ask it to write the plan first, then act).

**Why does it matter?**
For a multi-step goal ("find low items, then reorder them"), good step order is the difference between success and a confused mess. Reviewing the plan *before* acting is also your main safety lever for risky tasks.

**How does it work?**
- *Implicit:* in the loop you already built, each turn the model re-plans based on the latest observation. It naturally calls `list_items`, then `check_stock` per item — no extra code.
- *Explicit:* you first ask Claude to return a numbered plan (no tools), you read it, then you let it execute. Claude Code calls this **Plan mode** (Section 6).

> **Oracle DBA analogy (light):** explicit planning is like running **`EXPLAIN PLAN`** before a heavy query — you see the intended steps and can catch a bad one *before* it touches anything.


> **Remember this:** *Agents re-plan every turn from what they just saw — and for anything risky, make them show the plan before they act.*

**Don't mix these up**
- ❌ "The agent makes one plan at the start and follows it blindly." → ✅ It adapts the plan each turn based on new observations.
- ❌ "Planning and acting are the same step." → ✅ Separating them (plan first, approve, then act) is how you stay in control of risky work.

## 🧪 Lab D — Make the Plan Explicit (no tools)

**Objective:** get a reviewable plan *before* any action — the cheap, safe way to inspect an agent's intent.

In [ ]:
# Ask for a plan only. No tools passed, so it cannot act — it can only think.
plan = client.messages.create(
    model=MODEL,
    max_tokens=400,
    system="You are a careful planning agent. Output a short numbered plan only. Do not act.",
    messages=[{"role": "user", "content":
        "Goal: find every inventory item below 10 units and reorder each up to 50. "
        "List the exact steps and which tool you'd call at each step."}],
)
print(plan.content[0].text)

**Expected result:** a numbered plan like *1) list_items, 2) check_stock for each, 3) place_reorder for those below 10*. You can approve or correct it before letting the agent run.
**Try this:** add a deliberately tricky constraint ("but never reorder perishable items") and see how the plan changes.

---

# Section 5 — Safety: Permissions, Least Privilege & Limits

**What is it?**
Safety rails are the limits you put around the agent's *actions*: which tools it may use, when it must ask a human, and how many turns it may take.

**Why does it matter?**
An agent's tools touch the real world — files, databases, money. A confused agent with broad powers can do real damage. Rails turn "it can do anything" into "it can do exactly this much."

**The three rails you'll use most:**
1. **Least privilege** — give only the tools the task needs. A "summarise stock" agent gets `check_stock`/`list_items`, never `place_reorder`.
2. **Human-in-the-loop** — for any action that changes things or spends money, require an explicit human OK before running it.
3. **Turn limit** — cap the loop (the `range(5)` you already wrote) so a stuck agent can't loop forever and burn tokens.

> **Oracle DBA analogy (light):** least privilege is exactly **`GRANT`/`REVOKE`** — you give the account only the privileges it needs and nothing more. The human-OK step is like requiring a DBA sign-off before running a destructive `DROP`/`DELETE` in production.


**Real-world example.** A refunds agent at a retailer can *look up* any order freely, but issuing a refund over a threshold pauses for a human approval — least privilege plus human-in-the-loop.

> **Remember this:** *Give the agent the fewest tools that do the job, and make it ask a human before anything it can't undo.*

**Don't mix these up**
- ❌ "Safety means a smarter prompt." → ✅ Safety means *fewer tools* and *hard checks in your code* — not just nicer instructions.
- ❌ "A turn limit hurts the agent." → ✅ It protects you from runaway loops and surprise token bills.

## 🧪 Lab E — Add a Human-in-the-Loop Gate

**Objective:** wrap a "changes things" tool so it can't run without explicit approval.

In [7]:
# A risky tool: it changes state. We gate it behind a human OK.
def place_reorder(item, quantity):
    return f"Reordered {quantity} units of {item}."

def run_tool_safely(name, args):
    if name == "place_reorder":
        ok = input(f"Approve reorder {args['quantity']} x {args['item']}? (y/n) ")
        if ok.strip().lower() != "y":
            return "DENIED by human."
        return place_reorder(args["item"], args["quantity"])
    return run_tool(name, args)   # read-only tools from Lab C run freely

# Demo without the API: pretend Claude asked to reorder
print(run_tool_safely("place_reorder", {"item": "butter", "quantity": 48}))

Approve reorder 48 x butter? (y/n) n
DENIED by human.


**Expected result:** a prompt asks you to approve; only "y" lets the reorder run. Read-only tools still run freely.
**Try this:** add a rule that auto-approves reorders under 10 units but always asks for larger ones.

## ✅ Quick Check 3
> **Q:** You want an agent that can read inventory but must never reorder without you. Which rail does the work?
> **A:** **Least privilege** (don't even give it `place_reorder`) — and if it must have it, a **human-in-the-loop** gate before the call runs.

---

# Section 6 — The Same Loop, Productionised: Claude Code

You just built the agent loop by hand. **Claude Code is that exact loop, built and hardened by Anthropic**, running in your terminal with real tools that act on your computer.

**The mapping (this is the whole point):**

| What you built by hand | What Claude Code does |
|---|---|
| **Perceive** | Reads your goal + project files + last tool result |
| **Plan/Think** | Decides the next action (can show a plan) |
| **Act** | Calls a built-in tool: `Read`, `Write`, `Edit`, `Bash`, `Grep` |
| **Observe** | Reads the tool output (file contents, command output, errors) |
| **Repeat → stop** | Loops until the goal is met |

The big difference from your hand-built agent: Claude Code's tools act on **real files and your real shell**. That power is why its permission controls matter.

> **Where this runs:** in a **terminal on your computer** — not Colab. The cells below are commands you'll type there.


## Step 1 — Install & start
```bash
npm install -g @anthropic-ai/claude-code   # needs Node.js 18+; do NOT use sudo
mkdir agent-lab && cd agent-lab
claude                                      # first run opens a browser to sign in
```
At the prompt, try a warm-up: `What files are in this folder?` — watch it call a tool to look.

## Step 2 — Watch the loop recover from an error
Type:
```
Create calc.py with a function add(a, b), write a tiny test, run it, and fix anything that fails.
```
You'll see Write → Write test → Bash(run test) → if it fails, **Read error → Edit → re-run → passes**. That self-correction *is* your loop iterating.

## Step 3 — Plan mode (think before acting)
Press **Shift+Tab** to cycle to **plan mode** (or launch `claude --permission-mode plan`). Now the agent gets **read-only** tools — it can analyse and propose a plan but cannot change anything. This is the *explicit planning* from Section 4, built in.

> Same idea as **`EXPLAIN PLAN`**: see the intended steps before anything runs.

## Step 4 — Permissions = your safety rails, built in

| Mode | Behaviour | Use when |
|---|---|---|
| **default** | Asks before impactful actions | Normal supervised work |
| **plan** | Read-only, no changes | Reviewing before acting |
| **acceptEdits** | Auto-accepts file edits | You trust the task, want speed |
| **bypassPermissions** | No prompts at all | ⚠️ Throwaway sandboxes only |

This is your Section 5 lesson, productionised: permission modes are human-in-the-loop, and `--allowedTools` is least privilege.

## Step 5 — Headless mode: an agent in one command
Run it non-interactively with `-p`, and fix the toolset with `--allowedTools` (least privilege):
```bash
# read-only summary
claude -p "List the Python files here and summarise each" --allowedTools "Read,Grep"

# let it act, but only with the tools it needs
claude -p "Run the test suite and fix any failing tests" --allowedTools "Bash,Read,Edit"
```
`-p` runs the prompt, prints the result, and exits — perfect for scripts and CI.

## ✅ Quick Check 4
> **Q:** In Claude Code, what plays the role of your hand-built agent's turn limit + tool list (the safety rails)?
> **A:** **Permission modes** (default/plan/acceptEdits) and **`--allowedTools`** — the productionised version of human-in-the-loop and least privilege.

---

# Architecture — How a Production Agent Fits Together

Here's the end-to-end shape of the agent you built, drawn as a system an architect would review.

```mermaid
flowchart TD
    U[User goal] --> O[Agent loop / orchestrator]
    O -->|prompt + tools| C[Claude API]
    C -->|stop_reason=tool_use| O
    O -->|run requested tool| G{Permission gate}
    G -->|read-only: allow| T[Tools: check_stock, list_items]
    G -->|risky: ask human| H[Human approve y/n]
    H -->|approved| W[Tools: place_reorder]
    T --> O
    W --> O
    O -->|tool_result back to model| C
    C -->|stop_reason=end_turn| F[Final answer to user]
```

**Component responsibilities**
- **Orchestrator (your loop):** owns the `messages` list, the turn limit, and routing tool calls.
- **Claude API:** decides the next action; never executes tools itself.
- **Permission gate:** enforces least privilege and the human-in-the-loop check.
- **Tools:** the only things that touch the real world (data, files, money).

**Data flow:** goal → model → tool request → gate → tool runs → result back to model → repeat → final answer.

**Failure points & fixes:** runaway loop → *turn limit*; wrong/destructive action → *gate + least privilege*; tool error/timeout → *catch it and return the error text so the model can retry*; token blow-up → *small max_tokens, cheap model, short prompts*.

**Scaling / security / cost tradeoffs:** cache or batch tool reads; log every tool call for audit (a DBA would call this an **audit trail**); keep API keys in secrets, never in the notebook; pick Haiku unless a step truly needs deeper reasoning.


---

# 🛠️ Mini Project — Inventory Restock Agent

**Business use case.** A retailer wants an agent that, given a goal like *"keep us stocked,"* finds every item below a threshold and reorders it up to a target — but never places a large reorder without a human OK.

**Architecture.** The loop from Lab C + the safety gate from Lab E:
`list_items → check_stock per item → (if below threshold) place_reorder via human-gated tool → summarise what it did.`

**Build steps (Claude API):**
1. Reuse `STOCK`, the `tools` list, and `run_tool` from Lab C.
2. Add a `place_reorder(item, quantity)` tool definition + wire it into `run_tool_safely` from Lab E.
3. Give the agent one goal: *"Reorder every item below 10 units up to 50. Use the tools. Confirm large reorders with me."*
4. Run the loop; approve/deny when prompted; print the agent's final summary.

**Definition of done:** the agent autonomously identifies the low items (milk, bread, butter), proposes correct reorder quantities, pauses for your approval on each change, and ends with a clear summary — with no step-by-step steering from you.

**Stretch:** have it also write a `restock_report.csv`, then rebuild the same project in **Claude Code** by dropping `orders.csv` in a folder and giving it the goal — compare hand-built vs. productionised.


In [8]:
# Mini-project starter — combine Lab C tools + Lab E gate into one agent run.
# (Assumes client, MODEL, STOCK, tools, run_tool, run_tool_safely already defined above.)

tools = tools + [{
    "name": "place_reorder",
    "description": "Reorder a quantity of an item. Changes inventory, so confirm large amounts.",
    "input_schema": {
        "type": "object",
        "properties": {"item": {"type": "string"}, "quantity": {"type": "integer"}},
        "required": ["item", "quantity"],
    },
}]

messages = [{"role": "user", "content":
    "Reorder every item below 10 units up to a total of 50. Use the tools. Confirm large reorders."}]

for turn in range(8):
    reply = client.messages.create(model=MODEL, max_tokens=600, tools=tools, messages=messages)
    messages.append({"role": "assistant", "content": reply.content})
    if reply.stop_reason != "tool_use":
        print("FINAL:", reply.content[0].text); break
    results = []
    for block in reply.content:
        if block.type == "tool_use":
            print(f"  ACT → {block.name}({block.input})")
            output = run_tool_safely(block.name, block.input)   # gate enforces human OK
            print(f"  OBS ← {output}")
            results.append({"type": "tool_result", "tool_use_id": block.id, "content": str(output)})
    messages.append({"role": "user", "content": results})

  ACT → list_items({})
  OBS ← ['milk', 'bread', 'eggs', 'butter', 'rice']
  ACT → check_stock({'item': 'milk'})
  OBS ← 8
  ACT → check_stock({'item': 'bread'})
  OBS ← 3
  ACT → check_stock({'item': 'eggs'})
  OBS ← 40
  ACT → check_stock({'item': 'butter'})
  OBS ← 2
  ACT → check_stock({'item': 'rice'})
  OBS ← 25
  ACT → place_reorder({'item': 'milk', 'quantity': 42})
Approve reorder 42 x milk? (y/n) y
  OBS ← Reordered 42 units of milk.
  ACT → place_reorder({'item': 'bread', 'quantity': 47})
Approve reorder 47 x bread? (y/n) n
  OBS ← DENIED by human.
  ACT → place_reorder({'item': 'butter', 'quantity': 48})
Approve reorder 48 x butter? (y/n) y
  OBS ← Reordered 48 units of butter.
FINAL: **Reorder Summary:**

✅ **milk**: Successfully reordered 42 units (8 → 50)
❌ **bread**: Reorder DENIED by human (need approval for 47 units)
✅ **butter**: Successfully reordered 48 units (2 → 50)

The bread reorder for 47 units was denied and requires your approval. Would you like to approve th

---

# 📚 Revision Suite

## Session Summary
You learned that an agent is just an LLM in a **loop** with **tools**. You built that loop by hand against the live Claude API: define tools → call the model → while `stop_reason == "tool_use"` run the tool and feed the result back → stop on `end_turn`. You added planning (explicit `EXPLAIN PLAN`-style review) and safety rails (least privilege, human-in-the-loop, turn limit), then saw the same loop productionised as **Claude Code**.

## What You Learned Today — you can now…
- Explain why an agent ≠ a smarter model (it's the same model + loop + tools).
- Define a tool (name, description, input_schema) and run it from your code.
- Implement the full tool-use loop with the Claude API.
- Make planning explicit and review it before acting.
- Add least-privilege, human-in-the-loop, and turn-limit safety rails.
- Drive Claude Code as the productionised version of the same loop.


## 🧠 AI Architect Cheat Sheet

**The one rule:** while `stop_reason == "tool_use"`, run the tool, append a `tool_result`, loop again.

**Tool shape:**
```python
{"name": ..., "description": ..., "input_schema": {"type":"object","properties":{...},"required":[...]}}
```
**Loop skeleton:**
```python
for turn in range(LIMIT):
    reply = client.messages.create(model=MODEL, max_tokens=N, tools=tools, messages=messages)
    messages.append({"role":"assistant","content":reply.content})
    if reply.stop_reason != "tool_use": print(reply.content[0].text); break
    results = [{"type":"tool_result","tool_use_id":b.id,"content":str(run_tool(b.name,b.input))}
               for b in reply.content if b.type=="tool_use"]
    messages.append({"role":"user","content":results})
```
**Key facts:** `reply.content[0].text` = the answer · `stop_reason`: `end_turn` (done) / `tool_use` (wants a tool) · default model `claude-haiku-4-5-20251001`.

**Safety table:** least privilege (fewest tools) · human-in-the-loop (gate risky calls) · turn limit (no runaway).

**DBA quick map:** tool ≈ stored-proc signature · loop ≈ cursor FETCH loop · plan mode ≈ EXPLAIN PLAN · least privilege ≈ GRANT/REVOKE · tool logging ≈ audit trail.


## ⏱️ 5-Minute Revision Guide
1. **Agent = LLM + loop + tools.** The model decides; the loop keeps it going.
2. **The loop:** call model → if `tool_use`, run tool, send `tool_result` back → repeat → stop on `end_turn`.
3. **Tool = dict** with name, description, input_schema. Claude requests; your code runs it.
4. **Planning:** the agent re-plans each turn; for risky tasks, get an explicit plan first (Plan mode / EXPLAIN PLAN).
5. **Safety:** least privilege + human-in-the-loop + turn limit.
6. **Claude Code** = this exact loop, productionised, acting on real files — with permissions and `--allowedTools` as the rails.


## 🎯 Interview Preparation Notes
- *What makes something an agent?* An LLM run in a loop with tools, so it can take multiple actions and self-correct toward a goal.
- *Walk me through the tool-use loop.* Call the model with tools; if `stop_reason == "tool_use"`, run the requested tool(s), append `tool_result` blocks (tied to each `tool_use_id`), and call again; stop on `end_turn`.
- *How do you keep an agent safe?* Least privilege (only needed tools), human-in-the-loop on irreversible actions, and a turn limit; log every tool call for audit.
- *(Architecture)* Design an agent that restocks inventory but never overspends. → Read-only tools run freely; reorders go through a permission gate with a human approval above a threshold; turn limit + audit log.
- *(FDE)* A client worries the agent will "go rogue." → Demo Plan mode and `--allowedTools`: show that it must reveal its plan and can only touch the tools you grant.


## 📝 Assignment
- **Beginner:** change Lab C's question to "list every item and its stock," and confirm the agent calls `list_items` then `check_stock`.
- **Intermediate:** add a `total_inventory_value` tool (price × stock) and ask the agent to report the most valuable item.
- **Advanced:** add retry handling — if a tool returns an error string, let the loop continue so the model can adjust, and cap retries.
- **Project:** finish the Inventory Restock Agent so it reorders all low items with human approval and writes a `restock_report.csv`, then rebuild it in Claude Code.


## ✅ Assessment

**Multiple choice (10)**
1. An agent is best described as: (a) a bigger model (b) an LLM in a loop with tools (c) a database (d) a prompt template
2. Which `stop_reason` means Claude wants to call a tool? (a) end_turn (b) max_tokens (c) tool_use (d) stop_sequence
3. A tool definition must include: (a) name, description, input_schema (b) only a name (c) Python code (d) a model ID
4. Who actually executes a tool? (a) Claude (b) your code (c) the SDK automatically (d) the input_schema
5. The turn limit exists to: (a) speed up replies (b) prevent runaway loops (c) improve grammar (d) lower temperature
6. Least privilege means: (a) give every tool (b) give only the tools needed (c) disable tools (d) use the cheapest model
7. Plan mode in Claude Code gives the agent: (a) all tools (b) read-only tools (c) no tools (d) only Bash
8. You send a tool's output back as: (a) a new user question (b) a tool_result tied to the tool_use_id (c) a system prompt (d) a comment
9. Default lab model here: (a) opus (b) sonnet (c) haiku (d) gpt
10. Human-in-the-loop is most important for: (a) read-only lookups (b) irreversible/expensive actions (c) printing text (d) listing files

**Short answer (5)**
1. In one sentence, what turns a plain LLM call into an agent?
2. State the one rule of the agent loop.
3. Why is the tool *description* important?
4. Name the three safety rails covered today.
5. What is the Claude Code equivalent of EXPLAIN PLAN, and why use it?

**Scenario (3)**
1. Your agent keeps calling tools forever and never finishes. What two things do you check first?
2. A reorder agent must never spend over $1,000 without approval. How do you enforce it?
3. You want to summarise log files but fear the agent might edit them. How do you run it safely in Claude Code?


## 🔑 Answer Key

**MCQ:** 1‑b · 2‑c · 3‑a · 4‑b · 5‑b · 6‑b · 7‑b · 8‑b · 9‑c · 10‑b

**Short answer:**
1. Wrapping the model in a loop and giving it tools so it can take multiple actions toward a goal.
2. While `stop_reason == "tool_use"`, run the tool, send the `tool_result` back, and loop again.
3. It's how Claude decides *when* to use the tool — a vague description leads to wrong or missed calls.
4. Least privilege, human-in-the-loop, and a turn limit.
5. **Plan mode** (read-only) — it shows the intended steps before anything runs, so you can catch a bad action early.

**Scenario:**
1. The **turn limit** (is it set / too high?) and whether `stop_reason` is being checked correctly to break the loop.
2. Route reorders through a permission gate that requires explicit human approval above the $1,000 threshold (and/or don't grant the spend tool at all unless needed).
3. Run headless with read-only tools: `claude -p "summarise these logs" --allowedTools "Read,Grep"` — no Write/Edit/Bash, so it can't change the files.

---
✅ **Lab complete.** You built a goal-driven agent by hand, then met its productionised twin in Claude Code.

### 🔍 Resources (Official)
- Tool use (the loop, under the hood) — https://docs.claude.com/en/docs/build-with-claude/tool-use
- Messages API — https://docs.claude.com/en/api/messages
- Claude Code overview — https://docs.claude.com/en/docs/claude-code/overview
- Claude Code headless mode — https://docs.claude.com/en/docs/claude-code/headless
